As of now we are building software based on rule based logics like if-else,for,while to take decisions, but if we can able to call LLM through API then we can bring AI based Human like intelligence for decision Making.

Why Do we Need LangChain?

Every LLM provide came up with his own code, LangChain is a abstraction Framework where we can use different model with minimal code changes. It is LangChain ecosystem and bulitin Plugins which can easy AI based Model Building.

# API Key Loading

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


# Model Invokation

In [6]:
from langchain.chat_models import init_chat_model

llm_call=init_chat_model(model="groq:openai/gpt-oss-120b")
res=llm_call.invoke("Hello, how are you?")
res.content


"Hello! I'm doing great, thanks for asking. How can I assist you today?"

# Messages and Context Setup

In [7]:
from langchain.messages import HumanMessage,SystemMessage

Context_prompt=SystemMessage(content="You are a Mainframe Subject Matter Expert. Answer the questions")
User_prompt=HumanMessage(content="Explain about comp, comp2, comp3?")

full_prompt=[Context_prompt,User_prompt]

res=llm_call.invoke(full_prompt)

print(res.content,end="",flush=True)

## COBOL “USAGE”‑clause ‑ `COMP`, `COMP‑2` and `COMP‑3`

In COBOL the **USAGE** clause tells the compiler how a data item is to be stored in memory.  
The three most common “compressed” (i.e., non‑display) formats on IBM mainframes are:

| USAGE | Common name | Physical representation | Typical size (bytes) | Range / Precision | Typical use |
|-------|-------------|--------------------------|----------------------|-------------------|-------------|
| `COMP` (or `COMP‑4`) | Binary integer | Binary (two’s‑complement) | 2 bytes for `PIC 9(4)`, 4 bytes for `PIC 9(9)`, 8 bytes for `PIC 9(18)` (or larger) | Signed integer, range depends on byte length (e.g., 4‑byte: –2 147 483 648 … +2 147 483 647) | Arithmetic counters, indexes, amounts that never need decimal points |
| `COMP‑2` | Floating‑point (IEEE) | IEEE‑754 binary floating point (single or double) | 4 bytes (single) or 8 bytes (double) | Approx. 7 decimal digits (single) or 15‑16 decimal digits (double) | Scientific calculations, hig


We can see LLM are providing response which is a free form, but for applications we need LLMS to provide structured output, our application may have multiple LLM and this Output will be passed to other LLMs as input.

We have Pydantic Lib which can be used to get Structured response, Validation, Parsing, Conversion

# Structured Response with Pydantic

In [8]:
from pydantic import BaseModel,Field
from typing import List

class Mainframe_Bot(BaseModel):
    """ 
    Schema for Structured Response for the Mainframe Bot
    """
    response:str=Field(description="Response to the User Query")
    examples:List[str]=Field(description="List of Examples")

In [9]:
structured_llm=llm_call.with_structured_output(Mainframe_Bot)

res=structured_llm.invoke(full_prompt)

print(res.response)
print(res.examples)

### COBOL Computational Data Types

| Data‑type | Alias | Typical Size | Storage Format | Typical Use‑Case |
|-----------|-------|--------------|----------------|-----------------|
| **COMP**  | `BINARY` (IBM) | 2 bytes (PIC 9(4)), 4 bytes (PIC 9(9)), 8 bytes (PIC 9(18)) | Binary (two’s‑complement) integer | Fast integer arithmetic, counters, indexes |
| **COMP‑2**| `FLOAT` (IBM) | 4 bytes (single‑precision), 8 bytes (double‑precision) | IEEE‑754 floating‑point (or platform native) | Scientific/engineering calculations requiring fractional values |
| **COMP‑3**| `PACKED‑DECIMAL` | (n + 1)/2 bytes for n digits (rounded up) | Packed decimal (BCD) with sign nibble in last half‑byte | Financial data, exact decimal arithmetic, large numeric fields |

#### 1. **COMP (Binary Integer)**
- **Definition**: Stores numeric data in the machine’s native binary format (two’s‑complement). 
- **Declaration**: `PIC 9(4) COMP.` (2‑byte integer), `PIC 9(9) COMP.` (4‑byte integer), `PIC 9(18) COMP.` (8‑byt

# Memory, Agents, Middleware

We can append the AI response to prompt and can create Memory but we have inbuilt function in Langchain for memory.

Agent = Model + System Prompt +Tool Calling + Memory

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

mem=InMemorySaver()
Sum_mid= SummarizationMiddleware(model=llm_call,trigger={"messages":10},keep_recent = 3)

Mainframe_agent = create_agent(
    model=llm_call,
    system_prompt=Context_prompt,
    checkpointer= mem,
    middleware=[Sum_mid]
)

agent_conf={"configurable":{"thread_id":"Mainframe_bot"}}

res = Mainframe_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain about COMP, COMP-2, and COMP-3?"
        }
    ]
},config=agent_conf,checkpointer=mem)

print(res["messages"][-1].content)

## COBOL “COMPUTATIONAL” (COMP) Data‑Types – A Quick‑Reference Guide  

| **Data‑type** | **Common name** | **What it stores** | **Internal representation** | **Typical size (bytes)** | **Range / Precision** | **Typical use‑cases** |
|---------------|-----------------|--------------------|-----------------------------|--------------------------|-----------------------|------------------------|
| **COMP**      | Binary (signed integer) | Binary integer values (no decimal point) | Two’s‑complement binary, native to the machine | 2 bytes for PIC 9(4)‑9(5); 4 bytes for PIC 9(9)‑9(10); 8 bytes for PIC 9(18)‑9(19) (depends on compiler) | ±(2ⁿ⁻¹‑1) where *n* = bits (e.g., 2‑byte → ±32 767; 4‑byte → ±2 147 483 647) | High‑speed arithmetic, counters, indexes, flags, where no decimal fraction is required |
| **COMP‑2**    | Binary floating‑point | Approximate real numbers (fractional) | IEEE‑754 binary floating‑point (single‑ or double‑precision) – exact format varies by compiler (most IBM Enter

# Tool Integration

LLM comes with Knowledge Cutoff, so if we ingegrate LLM with Tools then LLM, then LLM can call tool when required

These tolls can be custom tools(@tool def) or inbuilt tools

In [11]:
from langchain_community.tools import DuckDuckGoSearchRun

websearch=DuckDuckGoSearchRun()

C:\Users\Admin\AppData\Local\Temp\ipykernel_2404\2466924142.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

mem = InMemorySaver()

Sum_mid = SummarizationMiddleware(
    model="groq:openai/gpt-oss-120b",
    trigger={"messages": 10},
    keep_recent=3
)

Mainframe_agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    system_prompt=Context_prompt,
    checkpointer=mem,
    middleware=[Sum_mid],
    tools=[websearch]
)

agent_conf = {
    "configurable": {
        "thread_id": "Mainframe_bot"
    }
}

res = Mainframe_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather Today in Palakonda?"
            }
        ]
    },
    config=agent_conf
)

print(res["messages"][-1].content)

**Palakonda (Andhra Pradesh, India) – Weather for September 8 2026 (today)**  

| Parameter | Value |
|-----------|-------|
| **General condition** | Mostly cloudy and hot |
| **Temperature** | High ≈ 94 °F (35 °C) – feels like 108 °F (42 °C) ; Low ≈ 84 °F (29 °C) |
| **Precipitation** | ~ 82 % chance of showers throughout the day |
| **Wind** | Light wind from the **WNW** at about **5 mph** (≈ 9 km/h) |
| **Heat risk** | **High** – strong heat‑related health warnings in effect |
| **Humidity** | Not listed in the snippet, but with a high chance of showers and a “hot, mostly‑cloudy” profile, relative humidity is typically in the 60‑80 % range for this time of year. |
| **Sunrise / Sunset** | Approx. sunrise ≈ 06:00 IST, sunset ≈ 18:00 IST (typical for early‑September in this region). |

**What this means for you today**

- Expect **very warm to hot** temperatures, especially in the afternoon, with the “feels‑like” temperature pushing well above 100 °F (≈ 42 °C).  
- **Cloud cover will 